# Notebook 36 -- EKF Baseline (21-state Augmented)

Extended Kalman Filter with Jacobian computed via `jax.jacobian`:
1. Verify convergence on L1 (healthy): all params -> 1.0.
2. Compare EKF vs SBI on all 12 scenarios.
3. Show EKF overconfidence near snowball (L10).
4. Coverage comparison: SBI 90% CI vs EKF 90% CI.


In [ ]:
import sys; sys.path.insert(0, '../src')
import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pickle

from cstr_sbi.luyben.ekf import LuybenEKF, run_ekf_on_window, PARAM_NAMES
from cstr_sbi.luyben.priors import PARAM_NAMES as PNAMES
from cstr_sbi.luyben.inference import sample_posterior
from cstr_sbi.luyben.summaries import compute_summary_statistics
from cstr_sbi.luyben.simulator import simulate_em_window, warm_start_ic, apply_sensor_layer
from cstr_sbi.luyben.physics import NOMINAL_INLET, NOMINAL_CTRL_ALL
from cstr_sbi.luyben.scenarios import list_closed_loop_configs

with open('../results/luyben_posterior.pkl', 'rb') as f:
    posterior = pickle.load(f)['posterior']


In [ ]:
# EKF on L1 (healthy) -- convergence check
from cstr_sbi.luyben.scenarios import SCENARIO_CONFIGS
sc1 = SCENARIO_CONFIGS['L1_healthy']
theta = sc1.theta()
y0 = warm_start_ic(theta)
proc_key = jax.random.PRNGKey(0)
_, ys, obs = simulate_em_window(theta, NOMINAL_INLET, NOMINAL_CTRL_ALL, y0, key=proc_key)
t_out = np.arange(1, obs.shape[0]+1) * 1.0

ekf_result = run_ekf_on_window(np.asarray(obs), t_out)
print('EKF L1 (healthy) final estimates (should be ~1.0 for first 7, ~0.0 for delta):')
for j, pname in enumerate(list(PNAMES)):
    print(f'  {pname:10s}: {ekf_result["final_mean"][j]:.4f} +/- {ekf_result["final_std"][j]:.4f}')


In [ ]:
# SBI vs EKF comparison across all scenarios
rows = []
param_names_list = list(PNAMES)

for sc in list_closed_loop_configs()[:6]:  # first 6 scenarios for speed in dev
    theta = sc.theta()
    y0 = warm_start_ic(theta)
    proc_key, sens_key = jax.random.split(jax.random.PRNGKey(sc.id * 99))
    _, _, obs = simulate_em_window(theta, NOMINAL_INLET, NOMINAL_CTRL_ALL, y0, key=proc_key)
    obs_noisy = apply_sensor_layer(obs, key=sens_key)
    t_out = jnp.arange(1, obs.shape[0]+1) * 1.0
    obs_np = np.asarray(obs_noisy)
    t_np   = np.asarray(t_out)

    # SBI
    s = np.asarray(compute_summary_statistics(obs_noisy, t_out))
    sbi_samples = sample_posterior(posterior, s, n_samples=5000)

    # EKF
    ekf_res = run_ekf_on_window(obs_np, t_np)

    for j, pname in enumerate(param_names_list):
        true_val = float(theta[j])
        rows.append({
            'scenario': sc.name,
            'param': pname,
            'true': true_val,
            'sbi_mean': float(np.mean(sbi_samples[:, j])),
            'sbi_bias': float(np.mean(sbi_samples[:, j])) - true_val,
            'sbi_ci90_width': float(np.percentile(sbi_samples[:, j], 95) - np.percentile(sbi_samples[:, j], 5)),
            'ekf_mean': float(ekf_res['final_mean'][j]),
            'ekf_bias': float(ekf_res['final_mean'][j]) - true_val,
            'ekf_ci90_width': float(2 * 1.645 * ekf_res['final_std'][j]),
        })

import pandas as pd
df = pd.DataFrame(rows)
print(df.groupby('param')[['sbi_bias', 'ekf_bias', 'sbi_ci90_width', 'ekf_ci90_width']].mean().round(4).to_string())


## Snowball (L10): EKF overconfidence

In [ ]:
sc10 = SCENARIO_CONFIGS['L10_snowball']
theta10 = sc10.theta()
y0_10 = warm_start_ic(theta10)
proc_key10 = jax.random.PRNGKey(1010)
_, _, obs10 = simulate_em_window(theta10, NOMINAL_INLET, NOMINAL_CTRL_ALL, y0_10, key=proc_key10)
t_out10 = jnp.arange(1, obs10.shape[0]+1) * 1.0

s10 = np.asarray(compute_summary_statistics(obs10, t_out10))
sbi_samples10 = sample_posterior(posterior, s10, n_samples=5000)
ekf_res10 = run_ekf_on_window(np.asarray(obs10), np.asarray(t_out10))

# Plot (alpha, eta_p) with EKF Gaussian ellipse overlay
from matplotlib.patches import Ellipse
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(sbi_samples10[:, 0], sbi_samples10[:, 4], alpha=0.05, s=3, c='C0', label='SBI samples')
ax.axvline(float(theta10[0]), c='C3', ls='--', label=f'true alpha={theta10[0]:.2f}')
ax.axhline(float(theta10[4]), c='C1', ls='--', label=f'true eta_p={theta10[4]:.2f}')
# EKF Gaussian (1 sigma)
ekf_a, ekf_ep = ekf_res10['final_mean'][0], ekf_res10['final_mean'][4]
std_a, std_ep  = ekf_res10['final_std'][0],  ekf_res10['final_std'][4]
ellipse = Ellipse((ekf_a, ekf_ep), width=2*1.645*std_a, height=2*1.645*std_ep,
                  edgecolor='C2', facecolor='none', lw=2, label='EKF 90% CI')
ax.add_patch(ellipse)
ax.set_xlabel('alpha'); ax.set_ylabel('eta_p')
ax.set_title('L10 Snowball: SBI posterior vs EKF Gaussian')
ax.legend()
plt.tight_layout()
plt.show()
